# Dashboard Interativo – Eficiência Energética e Mobilidade Elétrica

Este dashboard apresenta uma visualização interativa dos principais indicadores associados à modernização da iluminação pública e à viabilidade de integração de carregadores para veículos elétricos (VE).

A análise inclui:
- perfis horários de consumo antes e depois da modernização LED;
- capacidade instalada e disponível nos PTD;
- estimativa da potência libertada;
- cenários de integração de carregadores VE;
- mapa ilustrativo das zonas analisadas.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown

In [2]:
ip_data = pd.read_excel("../data/IP_data.xlsx")
ptd_data = pd.read_excel("../data/PTD_data.xlsx")

In [3]:
concelhos_demo = ["Aveiro", "Porto", "Lisboa", "Setúbal", "Braga", "Coimbra"]

ip_demo = ip_data[ip_data["Concelho"].isin(concelhos_demo)].copy()
ptd_demo = ptd_data[ptd_data["Concelho"].isin(concelhos_demo)].copy()

ip_demo["Is_Ineficiente"] = ip_demo["Tipo de Lâmpada"].isin(["Sódio", "Mercúrio"]).astype(int)
ip_demo["Potencia_kW"] = ip_demo["Potência Instalada Total (W)"] / 1000

ip_group = ip_demo.groupby(["CodDistritoConcelho", "Distrito", "Concelho"], as_index=False).agg(
    P_IP_Total=("Potencia_kW", "sum"),
    P_IP_Inef=("Potencia_kW", lambda x: x[ip_demo.loc[x.index, "Is_Ineficiente"] == 1].sum())
)

ptd_demo["Utilizacao_decimal"] = (
        ptd_demo["Nível de Utilização [%]"]
        .astype(str)
        .str.extract(r'(\d+)%$')[0]
        .astype(float) / 100
)

ptd_group = ptd_demo.groupby(["CodDistritoConcelho", "Concelho"], as_index=False).agg(
    Cap_PTD=("Potência instalada [kVA]", "sum"),
    Util_Media=("Utilizacao_decimal", "mean"),
    N_PTDs=("Código de Instalação", "count")
)

df_dashboard = pd.merge(
    ip_group,
    ptd_group,
    on=["CodDistritoConcelho", "Concelho"],
    how="inner"
)

df_dashboard["Delta_PLED"] = df_dashboard["P_IP_Inef"] * 0.65
df_dashboard["PFolga"] = (df_dashboard["Cap_PTD"] * 0.92) * (1 - df_dashboard["Util_Media"])
df_dashboard["PVE"] = df_dashboard["N_PTDs"] * 22 * 0.60
df_dashboard["D"] = df_dashboard["PFolga"] + df_dashboard["Delta_PLED"] - df_dashboard["PVE"]
df_dashboard["Rate_Ineficiencia"] = df_dashboard["P_IP_Inef"] / df_dashboard["P_IP_Total"]

df_dashboard

,CodDistritoConcelho,Distrito,Concelho,P_IP_Total,P_IP_Inef,Cap_PTD,Util_Media,N_PTDs,Delta_PLED,PFolga,PVE,D,Rate_Ineficiencia
0,105,Aveiro,Aveiro,1055.192001,144.420,197485,0.475475,509,93.87300,95298.999935,6718.8,88674.072935,0.136866
1,303,Braga,Braga,2502.533002,1320.345,374450,0.542346,893,858.22425,157659.019466,11787.6,146729.643716,0.527603
2,603,Coimbra,Coimbra,2777.530400,1714.755,305779,0.540572,734,1114.59075,129244.808311,9688.8,120670.599061,0.617367
3,1106,Lisboa,Lisboa,9065.655300,28.985,1767583,0.400514,2278,18.84025,974870.349246,30069.6,944819.589496,0.003197
4,1312,Porto,Porto,2526.374100,1030.520,837600,0.421909,1144,669.83800,445472.164893,15100.8,431041.202893,0.407905
5,1512,Setúbal,Setúbal,1250.224663,314.125,224490,0.523736,560,204.18125,98363.130462,7392.0,91175.311712,0.251255


In [4]:
coords = pd.DataFrame({
    "Concelho": ["Aveiro", "Porto", "Lisboa", "Setúbal", "Braga", "Coimbra"],
    "Latitude": [40.6405, 41.1579, 38.7223, 38.5244, 41.5454, 40.2033],
    "Longitude": [-8.6538, -8.6291, -9.1393, -8.8882, -8.4265, -8.4103]
})

df_dashboard = df_dashboard.merge(coords, on="Concelho", how="left")
df_dashboard

,CodDistritoConcelho,Distrito,Concelho,P_IP_Total,P_IP_Inef,Cap_PTD,Util_Media,N_PTDs,Delta_PLED,PFolga,PVE,D,Rate_Ineficiencia,Latitude,Longitude
0,105,Aveiro,Aveiro,1055.192001,144.420,197485,0.475475,509,93.87300,95298.999935,6718.8,88674.072935,0.136866,40.6405,-8.6538
1,303,Braga,Braga,2502.533002,1320.345,374450,0.542346,893,858.22425,157659.019466,11787.6,146729.643716,0.527603,41.5454,-8.4265
2,603,Coimbra,Coimbra,2777.530400,1714.755,305779,0.540572,734,1114.59075,129244.808311,9688.8,120670.599061,0.617367,40.2033,-8.4103
3,1106,Lisboa,Lisboa,9065.655300,28.985,1767583,0.400514,2278,18.84025,974870.349246,30069.6,944819.589496,0.003197,38.7223,-9.1393
4,1312,Porto,Porto,2526.374100,1030.520,837600,0.421909,1144,669.83800,445472.164893,15100.8,431041.202893,0.407905,41.1579,-8.6291
5,1512,Setúbal,Setúbal,1250.224663,314.125,224490,0.523736,560,204.18125,98363.130462,7392.0,91175.311712,0.251255,38.5244,-8.8882


In [5]:
horas = list(range(24))

consumo_antes = np.array([
    18, 18, 18, 18, 18, 20, 28, 35, 40, 38, 34, 30,
    28, 27, 28, 32, 38, 45, 52, 55, 53, 46, 34, 24
])

consumo_depois = consumo_antes * 0.35

df_horario = pd.DataFrame({
    "Hora": horas,
    "Antes da modernização": consumo_antes,
    "Depois da modernização LED": consumo_depois
})

In [6]:
distritos = ["Todos"] + sorted(df_dashboard["Distrito"].dropna().unique().tolist())

dropdown_distrito = widgets.Dropdown(
    options=distritos,
    value="Todos",
    description="Distrito:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

display(dropdown_distrito)

Dropdown(description='Distrito:', layout=Layout(width='300px'), options=('Todos', 'Aveiro', 'Braga', 'Coimbra'…

In [7]:
def mostrar_dashboard(distrito_selecionado):
    if distrito_selecionado == "Todos":
        df_filtrado = df_dashboard.copy()
    else:
        df_filtrado = df_dashboard[df_dashboard["Distrito"] == distrito_selecionado].copy()

    display(Markdown("## Visão Geral dos Indicadores"))

    resumo = pd.DataFrame({
        "Indicador": [
            "Potência IP Total (kW)",
            "Potência Ineficiente (kW)",
            "Potência Libertada LED (kW)",
            "Capacidade PTD (kVA)",
            "Folga da Rede",
            "Saldo Final de Viabilidade"
        ],
        "Valor": [
            df_filtrado["P_IP_Total"].sum(),
            df_filtrado["P_IP_Inef"].sum(),
            df_filtrado["Delta_PLED"].sum(),
            df_filtrado["Cap_PTD"].sum(),
            df_filtrado["PFolga"].sum(),
            df_filtrado["D"].sum()
        ]
    })

    display(resumo.style.format({"Valor": "{:.2f}"}))

    display(Markdown("## 1. Perfis horários de consumo da iluminação pública"))

    fig_horario = go.Figure()
    fig_horario.add_trace(go.Scatter(
        x=df_horario["Hora"],
        y=df_horario["Antes da modernização"],
        mode="lines+markers",
        name="Antes da modernização"
    ))
    fig_horario.add_trace(go.Scatter(
        x=df_horario["Hora"],
        y=df_horario["Depois da modernização LED"],
        mode="lines+markers",
        name="Depois da modernização LED"
    ))
    fig_horario.update_layout(
        title="Perfis horários de consumo",
        xaxis_title="Hora do dia",
        yaxis_title="Consumo estimado (kW)",
        template="plotly_white"
    )
    fig_horario.show()

    display(Markdown("## 2. Capacidade instalada e capacidade disponível nos PTD"))

    fig_cap = px.bar(
        df_filtrado,
        x="Concelho",
        y=["Cap_PTD", "PFolga"],
        barmode="group",
        title="Capacidade instalada vs capacidade disponível",
        labels={"value": "Valor", "variable": "Indicador"},
        template="plotly_white"
    )
    fig_cap.show()

    display(Markdown("## 3. Estimativa de potência libertada pelas medidas de eficiência"))

    fig_led = px.bar(
        df_filtrado,
        x="Concelho",
        y="Delta_PLED",
        title="Potência libertada estimada com substituição por LED",
        labels={"Delta_PLED": "Potência libertada (kW)"},
        template="plotly_white"
    )
    fig_led.show()

    display(Markdown("## 4. Cenários de integração de carregadores VE"))

    fig_ve = px.bar(
        df_filtrado,
        x="Concelho",
        y=["PVE", "D"],
        barmode="group",
        title="Carga VE projetada e saldo final de viabilidade",
        labels={"value": "Valor", "variable": "Indicador"},
        template="plotly_white"
    )
    fig_ve.show()

    display(Markdown("## 5. Mapa das zonas analisadas"))

    fig_map = px.scatter_map(
        df_filtrado,
        lat="Latitude",
        lon="Longitude",
        hover_name="Concelho",
        hover_data={
            "Distrito": True,
            "P_IP_Total": ":.2f",
            "Cap_PTD": ":.2f",
            "D": ":.2f"
        },
        zoom=5.5,
        height=500,
        title="Localização ilustrativa dos concelhos analisados"
    )
    fig_map.update_layout(mapbox_style="open-street-map")
    fig_map.show()

In [8]:
output = widgets.interactive_output(
    mostrar_dashboard,
    {"distrito_selecionado": dropdown_distrito}
)

display(dropdown_distrito, output)

Dropdown(description='Distrito:', layout=Layout(width='300px'), options=('Todos', 'Aveiro', 'Braga', 'Coimbra'…

Output()